# 07c — Truy xuất bằng mô hình ngữ nghĩa

### Câu hỏi của notebook này

`bm25` tìm theo từ trùng nhau. Nếu câu truy vấn viết *"xe hơi"* mà tài liệu
viết *"ô tô"* thì không có từ nào trùng, và `bm25` không thấy tài liệu đó.

Mô hình ngữ nghĩa (bi-encoder) biến mỗi tài liệu thành một vector, biến câu
truy vấn thành một vector, rồi tìm vector gần nhau. Hai cách diễn đạt cùng một
ý cho hai vector gần nhau kể cả khi không trùng chữ nào.

Câu hỏi: **cách đó có tìm được những đáp án mà `bm25` bỏ sót không?**

### Vì sao câu trả lời có thể là không

Notebook `03b` đã đo một thứ khiến kỳ vọng phải hạ xuống: **99,3% đáp án có ít
nhất một từ chung với câu truy vấn của nó** (4.351 trên 4.381). Phần mà cách
tìm theo từ khoá *về nguyên tắc* không với tới chỉ là 0,7%.

Nghĩa là mô hình ngữ nghĩa không mở ra một kho đáp án mới. Nó có thể giúp theo
hai cách khác:

| Cách giúp | Vì sao |
| --- | --- |
| Xếp đúng hơn | Có từ chung không có nghĩa là `bm25` xếp nó cao |
| Ứng viên khác nhau | Hai mô hình sai ở những chỗ khác nhau, gộp lại phủ rộng hơn |

Cách thứ hai là lý do `07d` tồn tại. Notebook này đo xem có đáng làm `07d` hay
không.

### Chi phí, nói thẳng

| | Xếp hạng lại (`07b`) | Ngữ nghĩa (notebook này) |
| --- | --- | --- |
| Chạy trên | 1000 ứng viên mỗi câu | **Toàn bộ 1.654.055 tài liệu** |
| Tỷ lệ thuận với | Số câu truy vấn | Kích thước kho |
| Ước lượng | vài chục phút | vài giờ |

Đây là bước đắt nhất của dự án. Mục 1 tính ra con số cụ thể trước khi chạy.

### Đầu ra

| Kết quả | Dùng ở đâu |
| --- | --- |
| Hệ thống truy xuất bằng vector | `07d` |
| Trần của danh sách gộp `bm25` ∪ ngữ nghĩa | Quyết định `07d` có đáng không |
| Số đáp án chỉ ngữ nghĩa tìm được | Trả lời câu hỏi ở đầu notebook |

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "src"))
import reteco as R
import pipeline as P

DATA = Path(r"D:\RETECO-project\reteco_data\track1_tempo")
SYSTEMS = Path("..") / "systems"
GPU_JOBS = Path("..") / "gpu_jobs"
CACHE = Path("results")
GPU_JOBS.mkdir(exist_ok=True)

P.configure(DATA, CACHE)
domains = P.domains()
gold_all = {d: R.load_qrels(DATA / d / "qrels_train.txt") for d in domains}
print(f"{len(domains)} domains")

In [ ]:
# --- Shared chart style -------------------------------------------------
# Same palette and helpers as every other notebook in the project, so a bar
# here means what a bar there means.
BLUE, ORANGE, TEAL, AMBER, PINK, VIOLET = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7")
SURFACE, INK, INK_SOFT, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_SOFT, "axes.labelsize": 9.5,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "font.size": 10,
    "figure.dpi": 120, "axes.linewidth": 0.9,
})


def finish(ax, title, subtitle=None, xlabel=None, ylabel=None,
           grid_axis="y", note=None):
    if subtitle:
        ax.set_title(subtitle, loc="left", pad=8, fontsize=9.5, color=INK_MUTED)
        ax.annotate(title, xy=(0, 1), xycoords="axes fraction",
                    xytext=(0, 24), textcoords="offset points",
                    ha="left", va="bottom", fontsize=12.5,
                    fontweight="bold", color=INK, annotation_clip=False)
    else:
        ax.set_title(title, loc="left", pad=12, fontsize=12.5,
                     fontweight="bold", color=INK)
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left" if grid_axis == "x" else "bottom"].set_color(AXIS)
    if grid_axis == "x":
        ax.spines["bottom"].set_visible(False)
    elif grid_axis:
        ax.spines["left"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    ax.tick_params(length=0)
    if note:
        ax.text(0, -0.34, note, transform=ax.transAxes, ha="left", va="top",
                fontsize=8.5, color=INK_MUTED)
    return ax


def label_bars(ax, bars, values, fmt="{:,.0f}", horizontal=True, pad=0.015):
    span = max(values) if len(values) else 1
    for bar, value in zip(bars, values):
        if horizontal:
            ax.text(bar.get_width() + span * pad,
                    bar.get_y() + bar.get_height() / 2, fmt.format(value),
                    va="center", ha="left", fontsize=8.5, color=INK_SOFT)
        else:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + span * pad, fmt.format(value),
                    ha="center", va="bottom", fontsize=8.5, color=INK_SOFT)


def legend_below(ax, ncol=2, y=-0.30):
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, y),
              ncol=ncol, fontsize=9.5, handlelength=1.1, handleheight=1.1,
              columnspacing=1.8)

---

## 1. Chi phí, tính trước khi chạy

Một phép nhân đơn giản, nhưng nó quyết định có nên chạy hay không.

In [ ]:
# --- The bill, before spending it ----------------------------------------
sizes = {}
for domain in domains:
    sizes[domain] = sum(1 for _ in open(DATA / domain / "documents.jsonl",
                                        encoding="utf-8"))
total_docs = sum(sizes.values())

# A small sentence-transformer on a mid-range GPU embeds on the order of
# 400-2000 short documents a second, depending on the card and how much of
# each document is kept.
print(f"documents to embed: {total_docs:,}")
print()
print(f"{'rate (docs/s)':>16}{'hours':>10}")
print("-" * 26)
for rate in (400, 1000, 2000):
    print(f"{rate:>16,}{total_docs / rate / 3600:>10.1f}")
print("-" * 26)
print()
print(f"{'domain':<12}{'documents':>12}{'share':>9}")
print("-" * 33)
for domain, n in sorted(sizes.items(), key=lambda kv: -kv[1]):
    print(f"{domain:<12}{n:>12,}{n / total_docs:>9.1%}")
print("-" * 33)
print()
print("The job is written per domain and the worker finishes one before")
print("starting the next, so this can be spread over several sessions.")

---

## 2. Trần lý thuyết: phần `bm25` không với tới

Trước khi chạy GPU, đo lại chặn trên. Notebook `03b` đã tính con số này; mục
này in lại vì nó là thứ quyết định kỳ vọng của cả notebook.

In [ ]:
# --- How much is out of lexical reach at all -----------------------------
reach_file = CACHE / "lexical_reach.json"
if reach_file.exists():
    reach = json.loads(reach_file.read_text(encoding="utf-8"))
    reachable = sum(v["reachable"] for v in reach.values())
    total_gold = sum(v["total"] for v in reach.values())
    print(f"{'domain':<12}{'gold':>8}{'shares a word':>16}{'share':>9}")
    print("-" * 45)
    for domain, v in sorted(reach.items(),
                            key=lambda kv: kv[1]["reachable"] / kv[1]["total"]):
        print(f"{domain:<12}{v['total']:>8}{v['reachable']:>16}"
              f"{v['reachable'] / v['total']:>9.1%}")
    print("-" * 45)
    print(f"{'all':<12}{total_gold:>8}{reachable:>16}"
          f"{reachable / total_gold:>9.1%}")
    print()
    print(f"{total_gold - reachable} gold documents ({1 - reachable / total_gold:.1%}) "
          f"share no word with their query.")
    print("Those are the only ones a word-matching model cannot reach in")
    print("principle. Everything else it CAN reach and may simply rank badly.")
else:
    print("results/lexical_reach.json missing; run notebook 03b.")

---

## 3. Xuất gói việc ngữ nghĩa

Khác với `07b`, gói này không chứa danh sách ứng viên nào có ý nghĩa — máy GPU
sẽ tự nhúng cả kho và tự tìm. Gói chỉ mang câu truy vấn và tên các nhóm.

In [ ]:
# --- Write the dense job --------------------------------------------------
DENSE_JOB = "dense_01"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DENSE_DEPTH = 1000

# Any existing run works as the carrier: the dense worker reads the query list
# and the domain list, and ignores the candidates.
carrier = P.run(P.load_system(SYSTEMS / "01_bm25_tuned.json"), split="train")

job = P.export_gpu_job(carrier, GPU_JOBS / DENSE_JOB, DENSE_JOB,
                       task="dense", split="train", depth=1,
                       with_text=False, model=DENSE_MODEL)
print()
print(f"folder size: "
      f"{sum(p.stat().st_size for p in (GPU_JOBS / DENSE_JOB).iterdir()) / 1e6:.1f} MB")

### Lệnh chạy trên máy GPU

```bash
pip install torch sentence-transformers

hf download DataScience-UIBK/RETECO-SemEval2027 --repo-type dataset \
    --local-dir reteco_data --include "track1_tempo/*"

python gpu_worker.py dense_01 --data reteco_data/track1_tempo --depth 1000
```

Gói này **bắt buộc** cần `--data`: việc của nó là nhúng cả kho, nên kho phải
có mặt.

Công cụ làm xong nhóm nào ghi nhóm đó ra file. Ngắt giữa chừng thì chạy lại
đúng lệnh trên, nó bỏ qua những nhóm đã xong.

In [ ]:
# --- Take it back ---------------------------------------------------------
dense_scores = GPU_JOBS / DENSE_JOB / "scores.jsonl"

if not dense_scores.exists():
    print(f"{dense_scores} is not here yet.")
    print("Run the worker on the GPU machine and copy scores.jsonl back.")
    dense_digest = None
else:
    P.import_gpu_scores(dense_scores, DENSE_JOB)
    dense_system = {
        "name": "dense",
        "note": f"{DENSE_MODEL}, whole corpus embedded, job {DENSE_JOB}",
        "retrieve": {"kind": "gpu_job", "job": DENSE_JOB, "depth": DENSE_DEPTH},
    }
    (SYSTEMS / "05_dense.json").write_text(
        json.dumps(dense_system, indent=2), encoding="utf-8")
    dense_digest = P.run(dense_system, split="train")
    P.score(dense_digest)
    print("Imported and scored.")

---

## 4. Ngữ nghĩa so với từ khoá

Hai cột phải đọc riêng:

- `nDCG@10` trả lời *dùng một mình thì có tốt hơn không*
- `ceiling` trả lời *làm tầng một cho bước xếp hạng lại thì có tốt hơn không*

Một mô hình **tìm giỏi mà xếp dở** thua ở cột đầu và thắng ở cột sau. Với dự án
này, cột sau quan trọng hơn, vì `07b` đã có bộ xếp hạng lại rồi.

In [ ]:
# --- Side by side ---------------------------------------------------------
if dense_digest:
    sparse_digest = P.run(P.load_system(SYSTEMS / "03_stage1_deep.json"),
                          split="train")
    P.score(sparse_digest)

    print(f"{'system':<16}{'nDCG@10':>10}{'MAP':>9}{'R@100':>9}"
          f"{'ceiling':>10}{'fit':>9}{'check':>9}{'topics':>9}")
    print("-" * 81)
    for label, digest in (("bm25 (sau)", sparse_digest), ("dense", dense_digest)):
        r = P.score(digest)
        print(f"{label:<16}{r['macro']['ndcg']:>10.4f}{r['macro']['map']:>9.4f}"
              f"{r['macro']['recall']:>9.4f}{r['macro']['ceiling']:>10.4f}"
              f"{r['fit']['ndcg']:>9.4f}{r['check']['ndcg']:>9.4f}"
              f"{r['n_topics']:>9}")
    print("-" * 81)
    print()
    P.compare(dense_digest, sparse_digest, part="fit")

In [ ]:
# --- What does dense find that bm25 misses -------------------------------
if dense_digest:
    sparse_run = P._load_run(sparse_digest)
    dense_run = P._load_run(dense_digest)

    K = 100
    only_dense = only_sparse = both = neither = 0
    per_domain_gain = {}

    for domain in domains:
        gold = gold_all[domain]
        gain = 0
        for qid, want in gold.items():
            s = {d for d, _ in sparse_run.get(domain, {}).get(qid, [])[:K]}
            v = {d for d, _ in dense_run.get(domain, {}).get(qid, [])[:K]}
            for doc in want:
                in_s, in_v = doc in s, doc in v
                if in_s and in_v:
                    both += 1
                elif in_s:
                    only_sparse += 1
                elif in_v:
                    only_dense += 1
                    gain += 1
                else:
                    neither += 1
        per_domain_gain[domain] = gain

    found_total = both + only_sparse + only_dense
    print(f"Gold documents inside the top {K} of each model:")
    print(f"{'found by both':<24}{both:>8}")
    print(f"{'only bm25':<24}{only_sparse:>8}")
    print(f"{'only dense':<24}{only_dense:>8}   <- what dense adds")
    print(f"{'neither':<24}{neither:>8}")
    print("-" * 32)
    print(f"{'total gold':<24}{both + only_sparse + only_dense + neither:>8}")
    print()
    if only_dense:
        print(f"Dense reaches {only_dense} gold documents bm25 does not, "
              f"{only_dense / max(found_total, 1):.1%} of everything found.")
        print("That number is the whole case for fusing the two lists in 07d.")
    else:
        print("Dense reaches nothing bm25 misses. Fusing would add candidates")
        print("without adding answers, so 07d has no case at this depth.")

In [ ]:
# --- The overlap, drawn ---------------------------------------------------
if dense_digest:
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2))

    ax = axes[0]
    parts = [("Ca hai tim thay", both, TEAL),
             ("Chi bm25", only_sparse, BLUE),
             ("Chi dense", only_dense, VIOLET),
             ("Khong ben nao", neither, GRID)]
    total_gold_docs = sum(p[1] for p in parts)
    left = 0.0
    for label, value, colour in parts:
        share = value / total_gold_docs
        ax.barh([0], [share], left=left, color=colour, height=0.5, label=label)
        if share > 0.06:
            ax.text(left + share / 2, 0, f"{share:.0%}", ha="center",
                    va="center", fontsize=9.5,
                    color=INK_SOFT if colour is GRID else SURFACE,
                    fontweight="bold")
        left += share
    ax.set_xlim(0, 1); ax.set_ylim(-0.7, 0.4); ax.set_yticks([])
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    legend_below(ax, ncol=2, y=-0.22)
    finish(ax, f"Dense them {only_dense / max(total_gold_docs, 1):.1%} so dap an",
           subtitle=f"Dap an nam trong top {K} cua tung mo hinh",
           grid_axis=None)

    ax = axes[1]
    order = sorted(per_domain_gain.items(), key=lambda kv: -kv[1])
    bars = ax.barh([d for d, _ in order], [g for _, g in order],
                   color=VIOLET, height=0.62)
    ax.invert_yaxis()
    ax.set_xlim(0, max(max(per_domain_gain.values()), 1) * 1.25)
    label_bars(ax, bars, [g for _, g in order], fmt="{:.0f}")
    finish(ax, "Dense giup nhieu nhat o nhom nao",
           subtitle="So dap an chi dense tim duoc",
           xlabel="so dap an", grid_axis="x")

    fig.tight_layout()
    plt.show()

---

## 5. Gộp hai danh sách có nâng trần không

Câu hỏi cuối của notebook. Nếu gộp ứng viên của hai mô hình mà trần không nhúc
nhích thì `07d` không có việc gì để làm.

Phép so phải công bằng về ngân sách: gộp hai danh sách 500 thì được 1000 ứng
viên, nên nó phải so với **một mô hình lấy 1000**, không phải với một mô hình
lấy 500.

In [ ]:
# --- Does merging raise the ceiling, at equal budget ---------------------
if dense_digest:
    BUDGET = 1000

    def ceiling_of(lists):
        per_domain = {}
        for domain in domains:
            gold = gold_all[domain]
            values = [R.rerank_ceiling(lists[domain][q], gold[q])
                      for q in lists.get(domain, {}) if q in gold]
            per_domain[domain] = float(np.mean(values)) if values else 0.0
        return float(np.mean(list(per_domain.values())))

    sparse_alone = {d: {q: [x for x, _ in h][:BUDGET]
                        for q, h in per_query.items()}
                    for d, per_query in sparse_run.items()}
    dense_alone = {d: {q: [x for x, _ in h][:BUDGET]
                       for q, h in per_query.items()}
                   for d, per_query in dense_run.items()}
    merged = {}
    for domain in domains:
        merged[domain] = {}
        for qid in sparse_run.get(domain, {}):
            a = [x for x, _ in sparse_run[domain][qid]][:BUDGET // 2]
            b = [x for x, _ in dense_run.get(domain, {}).get(qid, [])][:BUDGET // 2]
            merged[domain][qid] = list(dict.fromkeys(a + b))

    options = [
        (f"bm25 mot minh, {BUDGET}", ceiling_of(sparse_alone)),
        (f"dense mot minh, {BUDGET}", ceiling_of(dense_alone)),
        (f"gop hai, {BUDGET // 2} moi ben", ceiling_of(merged)),
    ]
    print(f"{'option':<28}{'ceiling':>10}")
    print("-" * 38)
    for label, value in options:
        print(f"{label:<28}{value:>10.4f}")
    print("-" * 38)

    best_label, best_ceiling = max(options, key=lambda o: o[1])
    alone = options[0][1]
    print()
    if best_label.startswith("gop") and best_ceiling - alone > 0.005:
        print(f"Merging wins by {best_ceiling - alone:+.4f} at the same budget.")
        print("07d has a case: fuse the two lists, then rerank the result.")
    else:
        print(f"Merging does not beat one model at the same budget "
              f"({best_ceiling - alone:+.4f}).")
        print("07d should test fusion for RANKING rather than for coverage,")
        print("or be skipped in favour of a better reranker.")

In [ ]:
# --- Record the outcome ---------------------------------------------------
if dense_digest:
    dense_verdict = {
        "job_id": DENSE_JOB,
        "model": DENSE_MODEL,
        "depth": DENSE_DEPTH,
        "documents_embedded": total_docs,
        "scores": {name: P.score(d)["macro"]
                   for name, d in (("dense", dense_digest),
                                   ("sparse", sparse_digest))},
        "overlap_at_k": {"k": K, "both": both, "only_sparse": only_sparse,
                         "only_dense": only_dense, "neither": neither},
        "per_domain_dense_only": per_domain_gain,
        "union_ceiling": {label: value for label, value in options},
    }
    (CACHE / "dense_summary.json").write_text(
        json.dumps(dense_verdict, indent=1), encoding="utf-8")
    print(f"Saved {CACHE / 'dense_summary.json'}")
    print()
    P.table()

---

## 6. Kết luận

### Ba con số của notebook này

| Câu hỏi | Đọc ở mục |
| --- | --- |
| Ngữ nghĩa dùng một mình có hơn từ khoá không | 4, cột `nDCG@10` |
| Nó có tìm được đáp án từ khoá bỏ sót không | 4, dòng "chỉ dense" |
| Gộp hai danh sách có nâng trần không | 5 |

### Vì sao kỳ vọng phải vừa phải

99,3% đáp án có từ chung với câu truy vấn. Phần mà cách tìm theo từ khoá về
nguyên tắc không với tới chỉ là 0,7%, nên mô hình ngữ nghĩa không thể mở ra một
kho đáp án lớn. Nếu nó giúp, phần giúp nằm ở chỗ xếp đúng hơn và ở chỗ hai mô
hình sai khác nhau.

Ghi rõ điều này từ đầu để khỏi diễn giải quá tay một mức tăng nhỏ.

### Bước tiếp theo

Notebook `07d` gộp hai danh sách bằng Reciprocal Rank Fusion rồi xếp hạng lại
danh sách đã gộp. Đó là hệ thống đầy đủ nhất mà dự án dựng được, và là ứng viên
chính cho bài nộp.